# Notebook 01: Environment & Configuration Management

**SignalBrief: Declarative Configuration, Environment Validation, and Path Resolution**

---

### Section 1: Title, Project Context & Objectives
This notebook focuses on the configuration layer of SignalBrief. To maintain domain independence and free-tier compatibility, the platform must never hardcode URLs, topics, thresholds, or API endpoints. Everything is driven by declarative YAML definitions loaded into strictly typed Pydantic models.



### Section 2: Learning & Business Objectives
- **Business Objective**: Enable rapid addition of new domains (e.g., Supply Chain, AI, Biotech) simply by adding a YAML file without altering application code.
- **Learning Objective**: Master schema validation with Pydantic v2, YAML parsing, environment isolation, and graceful fallback handling.



### Section 3: Research Questions & Methodology
1. *How can we enforce strict type validation and range checking on user and domain parameters?*
2. *How do we ensure reproducible path resolution across Windows, Linux, and macOS without hardcoded forward/backward slashes?*
3. *How do we manage secrets safely via `.env` without exposing them in notebook outputs or git commits?*

**Methodology**:
- Use `pydantic` schemas for domain, source, and pipeline configurations.
- Use `pathlib.Path` for cross-platform filesystem handling.
- Verify environment loading with `python-dotenv`.



### Section 4: Libraries & Dependencies



In [1]:
import sys
import os
from pathlib import Path
import yaml
import pydantic
from dotenv import load_dotenv

PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Pydantic version: {pydantic.__version__}")
print(f"PyYAML version: {yaml.__version__}")



Pydantic version: 2.13.4
PyYAML version: 6.0.2


*Interpretation:* Pydantic v2 and PyYAML are loaded. The environment is verified.



### Section 5: Input Data Description
We inspect `.env.example`, `configs/pipeline.yaml`, and domain configurations.



In [2]:
env_example = PROJECT_ROOT / ".env.example"
pipeline_file = PROJECT_ROOT / "configs" / "pipeline.yaml"
print(f"Pipeline config exists: {pipeline_file.exists()} ({pipeline_file.stat().st_size} bytes)")
print(f".env.example exists: {env_example.exists()} ({env_example.stat().st_size} bytes)")



Pipeline config exists: True (1014 bytes)
.env.example exists: True (969 bytes)


### Section 6: Step-by-Step Implementation

#### Step 6.1: Load and Validate Global Pipeline Configuration



In [3]:
from signalbrief.config.loader import load_pipeline_config

pipeline_cfg = load_pipeline_config(pipeline_file)
print(f"Loaded Pipeline: {pipeline_cfg.name} (v{pipeline_cfg.version})")
print(f" - Lookback window: {pipeline_cfg.collection.lookback_hours} hours")
print(f" - Min article length: {pipeline_cfg.preprocessing.min_article_length_words} words")
print(f" - Max developments: {pipeline_cfg.analytics.max_developments_per_report}")
print(f" - Report template: {pipeline_cfg.reporting.template_path}")



Loaded Pipeline: SignalBrief Daily Intelligence Pipeline (v1.0.0)
 - Lookback window: 48 hours
 - Min article length: 40 words
 - Max developments: 5
 - Report template: templates/reports/daily_report.html.jinja2


*Interpretation:* Pipeline parameters are parsed and strongly typed.



#### Step 6.2: Load Domain Configurations and Validate Constraints



In [4]:
from signalbrief.config.loader import load_domain_config, list_available_domains

domains = list_available_domains(PROJECT_ROOT / "configs" / "domains")
print(f"Discovered domains: {domains}")

for dom_id in domains:
    dom = load_domain_config(dom_id, PROJECT_ROOT / "configs" / "domains")
    print(f"Domain: {dom.name} [ID: {dom.id}]")
    print(f"  Keywords ({len(dom.keywords)}): {dom.keywords[:3]}...")
    print(f"  Subtopics ({len(dom.subtopics)}): {dom.subtopics[:3]}...")
    print(f"  Max developments: {dom.report.max_developments}")



Discovered domains: ['artificial_intelligence', 'finance', 'manufacturing', 'supply_chain']
Domain: Artificial Intelligence [ID: artificial_intelligence]
  Keywords (8): ['foundation models', 'open weights', 'LLM evaluation']...
  Subtopics (5): ['multimodal models', 'retrieval augmented generation', 'synthetic data']...
  Max developments: 5
Domain: Financial Technology & Markets [ID: finance]
  Keywords (8): ['central bank digital currency', 'interest rates', 'payment infrastructure']...
  Subtopics (5): ['monetary policy', 'cross-border payments', 'retail banking']...
  Max developments: 5
Domain: Manufacturing [ID: manufacturing]
  Keywords (8): ['industrial automation', 'smart manufacturing', 'Industry 4.0']...
  Subtopics (6): ['industrial AI', 'workforce analytics', 'predictive maintenance']...
  Max developments: 5
Domain: Supply Chain & Logistics [ID: supply_chain]
  Keywords (8): ['freight rates', 'container shipping', 'procurement']...
  Subtopics (5): ['maritime shipping', 

*Interpretation:* Each domain configuration conforms to `DomainConfig`.



#### Step 6.3: Load and Inspect Source Registries



In [5]:
from signalbrief.config.loader import load_sources_for_domain

mfg_sources = load_sources_for_domain("manufacturing", PROJECT_ROOT / "configs" / "sources")
print(f"Total active sources for Manufacturing: {len(mfg_sources)}")
for src in mfg_sources:
    print(f" - [{src.id}] {src.name} -> {src.feed_url} (Reliability: {src.reliability_score})")



Total active sources for Manufacturing: 6
 - [nist_manufacturing] NIST Manufacturing News -> https://www.nist.gov/news-events/news/rss.xml (Reliability: 0.95)
 - [manufacturing_dive] Manufacturing Dive -> https://www.manufacturingdive.com/feeds/news/ (Reliability: 0.9)
 - [industry_week] IndustryWeek -> https://www.industryweek.com/rss/all (Reliability: 0.88)
 - [manufacturing_net] Manufacturing.net -> https://www.manufacturing.net/rss (Reliability: 0.85)
 - [reuters_technology] Reuters Technology Feed -> https://www.reutersagency.com/feed/?taxonomy=best-topics&post_type=best (Reliability: 0.95)
 - [hacker_news_rss] Hacker News Frontpage RSS -> https://news.ycombinator.com/rss (Reliability: 0.8)


*Interpretation:* Public RSS feeds are registered with polling frequencies and reliability scores.



### Section 7: Output Interpretation & Visualizations
We display a summary table of the configured domain parameters and sources.



In [6]:
import pandas as pd

source_data = [
    {
        "ID": s.id,
        "Name": s.name,
        "Domain": s.domain_id,
        "Type": s.source_type,
        "Reliability": s.reliability_score,
        "Poll (min)": s.polling_frequency_minutes,
    }
    for s in mfg_sources
]
df_sources = pd.DataFrame(source_data)
display(df_sources) if "display" in globals() else print(df_sources.to_string())



                   ID                       Name         Domain Type  Reliability  Poll (min)
0  nist_manufacturing    NIST Manufacturing News  manufacturing  rss         0.95         360
1  manufacturing_dive         Manufacturing Dive  manufacturing  rss         0.90         240
2       industry_week               IndustryWeek  manufacturing  rss         0.88         240
3   manufacturing_net          Manufacturing.net  manufacturing  rss         0.85         360
4  reuters_technology    Reuters Technology Feed        general  rss         0.95         360
5     hacker_news_rss  Hacker News Frontpage RSS        general  rss         0.80         180


### Section 8: Errors, Limitations & Assumptions
- Missing configuration files raise explicit `FileNotFoundError`.
- Malformed YAML files or invalid schema types raise `pydantic.ValidationError`.
- Credentials must never be written in `pipeline.yaml` or domain configs; all secrets are resolved from `.env`.



### Section 9: Summary of Findings
- Configuration loaders and schema validators operate predictably and catch malformed inputs.
- The system supports multi-domain extensibility with zero code modification.



### Section 10: Next Notebook's Input and Expected Output
- **Next Notebook**: `02_source_discovery.ipynb`
- **Input**: Approved sources from `configs/sources/manufacturing.yaml`.
- **Expected Output**: HTTP connectivity check, RSS feed reachability report, feed schema validation, and health benchmark.

